# Mood Classifier Training

**Label encoding (hardcoded):** `energetic=0, chill=1, sad=2, intense=3`

**Input:** `dataset.csv` — 57 audio features + `mood` column (from `ml/extract_features.py`)

**Output:**
- `mood_classifier.tflite` — deploy to `backend/models/`
- `scaler_params.json` — deploy to `backend/models/`

**Steps:** Run All → download both output files → copy to Pi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/dataset.csv')
print(f'Loaded {len(df)} rows, {df.shape[1]} columns')
print(df['mood'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

LABEL_MAP = {'energetic': 0, 'chill': 1, 'sad': 2, 'intense': 3}

feature_cols = [c for c in df.columns if c != 'mood']
X = df[feature_cols].values.astype(np.float32)
y = df['mood'].map(LABEL_MAP).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
import json
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

scaler_params = {
    'mean': scaler.mean_.tolist(),
    'scale': scaler.scale_.tolist()
}
with open('scaler_params.json', 'w') as f:
    json.dump(scaler_params, f)
print('Scaler params saved to scaler_params.json')

In [ ]:
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Input(shape=(57,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(4, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    validation_split=0.2,
    batch_size=16,
    verbose=1
)

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {accuracy:.2%}')

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('mood_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
print('Saved mood_classifier.tflite')

In [ ]:
from google.colab import files
files.download('mood_classifier.tflite')
files.download('scaler_params.json')
print('Download both files and copy to backend/models/ on the Pi.')